# Accumulation Offset Rescan

This notebook tests hypothetical software accumulation offsets for numbered `sync-check` runs without rerunning hardware. It needs saved event arrays plus `triggers.csv`. The sync sweep notebook enables `--save-filtered-events`, which writes `filtered_events.npz` even when `--event-noise-filter none` is selected.

If a run has only `raw.aedat4`, this notebook will skip it. Parsing AEDAT directly is intentionally not included here.

In [ ]:
from pathlib import Path
import csv
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUN_ROOT = Path("runs/camera")
offset_us_values = list(range(-3000, 3001, 100))
RUN_ROOT.resolve(), offset_us_values[:5], offset_us_values[-5:]

In [ ]:
def load_json(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def load_triggers(path):
    rows = []
    if not path.exists():
        return pd.DataFrame(columns=["index", "timestamp", "edge"])
    with path.open(encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            rows.append({
                "index": int(row["index"]),
                "timestamp": int(row["timestamp"]),
                "edge": row.get("edge", "rising"),
            })
    return pd.DataFrame(rows)


def load_event_npz(run_dir):
    for name in ("filtered_events.npz", "events.npz"):
        path = run_dir / name
        if path.exists():
            data = np.load(path)
            return path, {
                "x": np.asarray(data["x"]),
                "y": np.asarray(data["y"]),
                "t": np.asarray(data["t"], dtype=np.int64),
                "p": np.asarray(data["p"]),
            }
    return None, None


def window_counts(timestamps, starts, window_us):
    timestamps = np.sort(np.asarray(timestamps, dtype=np.int64))
    starts = np.asarray(starts, dtype=np.int64)
    if len(starts) == 0 or len(timestamps) == 0 or window_us <= 0:
        return np.zeros(len(starts), dtype=np.int64)
    ends = starts + int(window_us)
    left = np.searchsorted(timestamps, starts, side="left")
    right = np.searchsorted(timestamps, ends, side="left")
    return right - left

In [ ]:
def rescan_offsets(run_dir, offset_us_values, window_us=None):
    run_dir = Path(run_dir)
    summary = load_json(run_dir / "summary.json")
    metadata = load_json(run_dir / "metadata.json")
    triggers = load_triggers(run_dir / "triggers.csv")
    event_path, events = load_event_npz(run_dir)
    if events is None or triggers.empty:
        return pd.DataFrame()

    resolved_window_us = int(window_us or summary.get("window_us") or metadata.get("accumulation_window_us") or 0)
    rising = triggers[triggers["edge"].str.lower().eq("rising")].copy()
    trigger_timestamps = rising["timestamp"].to_numpy(dtype=np.int64)
    event_timestamps = events["t"]

    rows = []
    for offset_us in offset_us_values:
        starts = trigger_timestamps + int(offset_us)
        counts = window_counts(event_timestamps, starts, resolved_window_us)
        pre_counts = window_counts(event_timestamps, starts - resolved_window_us, resolved_window_us)
        post_counts = window_counts(event_timestamps, starts + resolved_window_us, resolved_window_us)
        in_sum = float(np.sum(counts))
        pre_sum = float(np.sum(pre_counts))
        post_sum = float(np.sum(post_counts))
        score = np.log1p(in_sum) - 1.25 * (pre_sum / (in_sum + 1.0)) - 0.75 * (post_sum / (in_sum + 1.0))
        rows.append({
            "run_dir": str(run_dir),
            "run_name": run_dir.name,
            "event_file": str(event_path),
            "window_us": resolved_window_us,
            "offset_us": int(offset_us),
            "trigger_count": int(len(trigger_timestamps)),
            "in_window_sum": in_sum,
            "pre_window_sum": pre_sum,
            "post_window_sum": post_sum,
            "in_window_mean": float(np.mean(counts)) if len(counts) else 0.0,
            "score": float(score),
        })
    return pd.DataFrame(rows)

In [ ]:
all_scans = []
skipped = []
for summary_path in sorted(RUN_ROOT.rglob("summary.json")):
    run_dir = summary_path.parent
    scan = rescan_offsets(run_dir, offset_us_values)
    if scan.empty:
        skipped.append(str(run_dir))
    else:
        all_scans.append(scan)

if all_scans:
    offset_scan_df = pd.concat(all_scans, ignore_index=True)
else:
    offset_scan_df = pd.DataFrame()

print(f"Scanned {len(all_scans)} runs; skipped {len(skipped)} runs without filtered_events.npz/events.npz")
offset_scan_df.head()

In [ ]:
if offset_scan_df.empty:
    print("No event NPZ files found. Re-run a small capture with --event-noise-filter local-support --save-filtered-events, then rerun this notebook.")
else:
    best_offset_by_run = (
        offset_scan_df.sort_values("score", ascending=False)
        .groupby("run_name", as_index=False)
        .first()
        .sort_values("score", ascending=False)
    )
    display(best_offset_by_run.head(20))

In [ ]:
if not offset_scan_df.empty:
    for run_name, group in offset_scan_df.groupby("run_name"):
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(group["offset_us"], group["score"], label="score")
        ax.plot(group["offset_us"], np.log1p(group["in_window_sum"]), label="log in-window")
        ax.set_title(run_name)
        ax.set_xlabel("offset_us")
        ax.set_ylabel("score")
        ax.legend()
        plt.show()
        if len(offset_scan_df["run_name"].unique()) > 8:
            break

In [ ]:
if not offset_scan_df.empty:
    Path("runs").mkdir(exist_ok=True)
    offset_scan_df.to_csv("runs/timing_offset_rescan_all.csv", index=False)
    best_offset_by_run.to_csv("runs/timing_offset_rescan_best.csv", index=False)
    print("Wrote runs/timing_offset_rescan_all.csv")
    print("Wrote runs/timing_offset_rescan_best.csv")